# Download Alpaca Dataset for SFT
Downloads Stanford Alpaca instruction-following examples and saves as structured JSONL to `datasets_sft/alpaca/`.
Each line preserves the instruction/response boundary needed for label masking during SFT.
Run `tokenize_sft` → `sft_alpaca` after.

In [ ]:
!pip install -q datasets

In [ ]:
import os
import json
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
OUT_DIR = os.path.join(SPARKYLLM, "datasets_sft", "alpaca")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = os.path.join(OUT_DIR, "alpaca_raw.jsonl")

# Check if already downloaded
if os.path.exists(OUT_FILE):
    size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
    print(f"Already exists: {OUT_FILE} ({size_mb:.1f} MB)")
    print("Delete the file and re-run to re-download.")
else:
    print("Loading tatsu-lab/alpaca from HuggingFace...")
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    print(f"Loaded {len(ds):,} examples")

    written = 0
    skipped = 0

    with open(OUT_FILE, "w", encoding="utf-8") as f:
        for ex in ds:
            instruction = ex["instruction"].strip()
            inp = ex["input"].strip()
            output = ex["output"].strip()

            if not instruction or not output:
                skipped += 1
                continue

            obj = {"instruction": instruction, "input": inp, "output": output}
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            written += 1

    # Save metadata
    meta = {"source": "tatsu-lab/alpaca", "num_examples": written, "skipped": skipped}
    meta_path = os.path.join(OUT_DIR, "meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
    print(f"\nDone!")
    print(f"  {written:,} examples written, {skipped:,} skipped")
    print(f"  {size_mb:.1f} MB")
    print(f"  Saved to: {OUT_FILE}")